# Data Generator
## 2.1 Setup Matrix Generator

In [1]:
import pandas as pd
import numpy as np

In [8]:
def generate_setup_matrix(products, machines):
    rows = []

    for m in machines['machine_id']:
        for p1 in products['product_id']:
            for p2 in products['product_id']:
                if p1 == p2:
                    setup = 0
                else:
                    setup = np.random.randint(5, 60)

                rows.append({
                    'from_product': p1,
                    'to_product': p2,
                    'machine_id': m,
                    'setup_time_min': setup
                })

    df = pd.DataFrame(rows)
    df.to_csv('setup_matrix.csv', index=False)
    return df

## 2.2 Machine Calendar Generator

In [10]:
from datetime import datetime, timedelta
def generate_machine_calendar(machines, days=5):
    from datetime import datetime, timedelta
    import numpy as np

    rows = []
    base_date = datetime(2025, 1, 1)

    for m in machines['machine_id']:
        for d in range(days):
            day = base_date + timedelta(days=d)

            # Define shifts safely
            shifts = [
                (8, 16),
                (16, 24)  # will handle 24 properly below
            ]

            for start_hr, end_hr in shifts:
                start_time = day.replace(hour=start_hr)

                if end_hr == 24:
                    end_time = (day + timedelta(days=1)).replace(hour=0)
                else:
                    end_time = day.replace(hour=end_hr)

                rows.append({
                    'machine_id': m,
                    'start_time': start_time,
                    'end_time': end_time,
                    'is_available': 1
                })

            # Optional downtime
            if np.random.rand() < 0.2:
                rows.append({
                    'machine_id': m,
                    'start_time': day.replace(hour=12),
                    'end_time': day.replace(hour=14),
                    'is_available': 0
                })

    df = pd.DataFrame(rows)
    df.to_csv('machine_calendar.csv', index=False)
    return df

## 2.3 Sections Generator

In [2]:
def generate_sections(num_sections=5):
    data = []

    for i in range(num_sections):
        data.append({
            'section_id': f'SEC_{i+1}',
            'description': f'Section {i+1}',
            'max_wip': np.random.randint(5, 20)
        })

    df = pd.DataFrame(data)
    df.to_csv('sections.csv', index=False)
    return df

## 2.4 Buffers Generator

In [3]:
def generate_buffers(sections):
    data = []

    for sec in sections['section_id']:
        data.append({
            'buffer_id': f'BUF_{sec}',
            'section_id': sec,
            'capacity': np.random.randint(5, 25)
        })

    df = pd.DataFrame(data)
    df.to_csv('buffers.csv', index=False)
    return df

## 2.5 Update Machines (capacity + section)

In [4]:
def enrich_machines(machines, sections):
    machines = machines.copy()

    machines['capacity'] = np.random.randint(1, 4, size=len(machines))
    machines['section_id'] = np.random.choice(sections['section_id'], size=len(machines))

    machines.to_csv('machines_updated.csv', index=False)
    return machines

## 2.6 Update Routing (buffers + setup family)

In [5]:
def enrich_routing(routing, buffers):
    routing = routing.copy()

    routing['setup_family'] = routing['product_id']  # simple version
    routing['buffer_id'] = np.random.choice(buffers['buffer_id'], size=len(routing))
    routing['transfer_time_min'] = np.random.randint(1, 15, size=len(routing))

    routing.to_csv('routing_updated.csv', index=False)
    return routing

## Main Data Generation Script

In [11]:
import pandas as pd
import numpy as np

# Import all generators (assuming same file or modularized)
# from generators import *

def main():
    print("=== Phase 1.5 Dummy Data Generation Started ===")

    # -----------------------------
    # 1. Load base data
    # -----------------------------
    print("\n1. Loading base CSVs...")
    machines = pd.read_csv('machines.csv')
    products = pd.read_csv('products.csv')
    routing = pd.read_csv('routing.csv')

    print(f"   Machines: {len(machines)}")
    print(f"   Products: {len(products)}")
    print(f"   Routing rows: {len(routing)}")

    # -----------------------------
    # 2. Generate Sections
    # -----------------------------
    print("\n2. Generating sections...")
    sections = generate_sections(num_sections=5)
    print(f"   Sections created: {len(sections)}")

    # -----------------------------
    # 3. Generate Buffers
    # -----------------------------
    print("\n3. Generating buffers...")
    buffers = generate_buffers(sections)
    print(f"   Buffers created: {len(buffers)}")

    # -----------------------------
    # 4. Enrich Machines
    # -----------------------------
    print("\n4. Enriching machines with capacity + sections...")
    machines_updated = enrich_machines(machines, sections)
    print("   machines_updated.csv created")

    # -----------------------------
    # 5. Enrich Routing
    # -----------------------------
    print("\n5. Enriching routing with buffers + setup family...")
    routing_updated = enrich_routing(routing, buffers)
    print("   routing_updated.csv created")

    # -----------------------------
    # 6. Generate Setup Matrix
    # -----------------------------
    print("\n6. Generating setup matrix...")
    setup_matrix = generate_setup_matrix(products, machines_updated)
    print(f"   Setup matrix rows: {len(setup_matrix)}")

    # -----------------------------
    # 7. Generate Machine Calendar
    # -----------------------------
    print("\n7. Generating machine calendar...")
    machine_calendar = generate_machine_calendar(machines_updated, days=5)
    print(f"   Calendar rows: {len(machine_calendar)}")

    # -----------------------------
    # 8. Final Summary
    # -----------------------------
    print("\n=== Data Generation Completed ===")
    print("\nGenerated Files:")
    print(" - sections.csv")
    print(" - buffers.csv")
    print(" - machines_updated.csv")
    print(" - routing_updated.csv")
    print(" - setup_matrix.csv")
    print(" - machine_calendar.csv")

    print("\n⚠️ Next Step: Integrate these into scheduler logic (setup + calendar aware engine)")



# -----------------------------
# Entry point
# -----------------------------
if __name__ == "__main__":
    main()

=== Phase 1.5 Dummy Data Generation Started ===

1. Loading base CSVs...
   Machines: 34
   Products: 150
   Routing rows: 1133

2. Generating sections...
   Sections created: 5

3. Generating buffers...
   Buffers created: 5

4. Enriching machines with capacity + sections...
   machines_updated.csv created

5. Enriching routing with buffers + setup family...
   routing_updated.csv created

6. Generating setup matrix...
   Setup matrix rows: 765000

7. Generating machine calendar...
   Calendar rows: 373

=== Data Generation Completed ===

Generated Files:
 - sections.csv
 - buffers.csv
 - machines_updated.csv
 - routing_updated.csv
 - setup_matrix.csv
 - machine_calendar.csv

⚠️ Next Step: Integrate these into scheduler logic (setup + calendar aware engine)
